# RV DeepSDF Training (Task 4.5 / 4.6 / 4.7)**Status: WRITTEN, NOT EXECUTED.** No cell has been run. Every shape, count and convergenceclaim below is design intent, not measurement. Section 8 lists what must pass first.Companion to `rv_4d_reconstruction.ipynb` (validated geometric baseline, Task 4.3).## Task 4.5 is resolved: the upstream trainer exists`https://github.com/yuan-xiaohan/4D-Myocardium-Reconstruction-with-Decoupled-Motion-and-Shape-Model`It ships `train_4dmm.py`, `process_data.py`, `reconstruct_4dmm.py`, config files and twotrained ACDC checkpoints (`endo`, `epi`). This notebook does **not** reimplement training —it prepares RV data in the upstream format and invokes the upstream trainer.An earlier draft of this notebook reconstructed the training loop from the components in`deep_sdf/`. That draft was discarded once the upstream repo was available: its guessedhyperparameters were wrong by large factors (`NumEpochs` 2000 vs the real 50; pointwiseweight 1e-3 vs 5e-4), and its loss terms did not match. Where this notebook and the upstreamrepo disagree, the upstream repo is right.

## 1. The two decoders are checkpoint-compatible (verified)Production and upstream ship **different source files for the same network**:| | production | upstream ||---|---|---|| file | `app/dependencies/networks/decoder.py` | `networks/4dmm_decoder.py` || `NetworkArch` | `"decoder"` | `"4dmm_decoder"` || specs kwargs | `motionmodel_kargs` / `shapemodel_kargs` | `motionnet_kargs` / `shapenet_kargs` || classes | `MotionModel` / `ShapeModel` | `MotionNet` / `ShapeNet` || tensor layout | flat `[M,3]`, per-point `t`/codes | batched `[b,N,3]`, per-sample `t`/codes || deformation scatter | Python loop over points (`decoder.py:163`) | `index_select` / `index_copy` |What matters is that **both assign `self.motion_net` and `self.shape_net`**, and the layerattribute names (`fc0..fc5`, `lin0..lin8`) are identical. `state_dict()` keys therefore matchexactly. Upstream's file even carries the comment *"Retain the original attribute layout;ModuleList would alter checkpoint keys"* — the compatibility is deliberate, not accidental.Confirmed by inspecting `examples/acdc/4DMM/endo/ModelParameters/latest.pth`:`{"epoch": 44, "model_state_dict": ...}`, 38 tensors, keys `motion_net.fc0.weight` ...`shape_net.lin8.bias`, **8.68 MB** — matching the 8.69 MB of production's`fourd_model_epoch_250.pth`.**Consequence:** a checkpoint trained by the upstream trainer loads into the production`FourDReconstructionHandler` unmodified. The differing `NetworkArch` / kwarg names live onlyin `specs.json`, and production supplies its own via `_get_default_specs()`.**This was executed, not assumed.** Building production's `Decoder` from production's ownspecs and calling `load_state_dict(strict=True)` on upstream's shipped `endo` checkpointsucceeds (epoch 44), and a forward pass returns `coords [16,3]`, `sdf [16,1]`. Section 7keeps the check runnable so it can be re-run against the RV checkpoint.

## 2. Two findings that change the plan### 2a. Production inference never runs the motion model`_transform_to_canonical_in_memory` returns a hardcoded `'t': 0.0`(`fourdreconstruction_handler.py:451`). That flows into `_optimize_latent_codes_sync`, whichbuilds an all-zero `t` and calls `decoder(xyz, t, c_m, c_s)`. Inside `Decoder.Deformation`:    index_nonED = torch.nonzero(t).squeeze().tolist()Empty for all-zero `t`, so `motion_net` is never evaluated. Each frame is insteadreconstructed independently by re-optimizing its own `c_s`(`_predict_4d_sequence` loops frames, calling `_optimize_latent_codes_sync` per frame).**Confirmed by execution, not only by reading:** calling the production `Decoder` with`t = zeros(16)` returns coordinates *bit-identical* to its input(`torch.equal(new_xyz, xyz) -> True`). The deformation branch is genuinely dead on this path.So a 4D checkpoint stays loadable and correct under the current inference path — `shape_net`is exercised normally, `motion_net` is simply unused — but the temporal coherence the modelis trained for is unreachable without an inference-side change (feed real `t`, one `c_s` persequence, one `c_m` per frame). Upstream's `reconstruct_4dmm.py` shows what that path lookslike. This is a new integration task, adjacent to the deferred 4.13-4.15, and should go tothe team: per-frame independent `c_s` is exactly what produces frame-to-frame jitter.### 2b. Variable-length sequences are supported upstreamProduction's bundled `deep_sdf/dataset.py` is an **older** revision: `SDFSamples` there isper-sequence, `torch.stack`s all frames, and needs a constant frame count. Upstream's isper-frame and manifest-driven (docstring: *"Dataset helpers for variable-length cardiacsequences"*), with `c_m = Embedding(len(dataset))` indexed globally and `frame_num` read fromeach sequence's `instance_list`.So patients with differing frame counts need **no temporal resampling**. patient005's 30frames and the LV default's 25 can sit in one corpus. Use the upstream dataset, not thebundled one.

## 3. Configuration

In [ ]:
from pathlib import Pathimport sys, os, json, math, subprocess# --- Upstream trainer (clone target) ---------------------------------------------UPSTREAM_URL  = "https://github.com/yuan-xiaohan/4D-Myocardium-Reconstruction-with-Decoupled-Motion-and-Shape-Model.git"UPSTREAM_DIR  = Path(r"E:\Jy\visheart-rv-4d-notebook\upstream_4dmm")# --- Production tree (for get_T and the compatibility assertion) ------------------REPO_ROOT   = Path(r"E:\Jy\COS40005FYPA")INFER_ROOT  = REPO_ROOT / "cardiac-component-segmentation-ai" / "visheart-inference-gpu"DEPS_DIR    = INFER_ROOT / "app" / "dependencies"# --- This notebook ----------------------------------------------------------------NB_ROOT    = Path(r"E:\Jy\visheart-rv-4d-notebook")MASK_PATHS = [NB_ROOT / "patient005_4d_segmentation.nii.gz"]   # append new patients hereSURFACE    = "rv"          # upstream ships "endo" and "epi"; RV is a third surfacePROC_ROOT  = UPSTREAM_DIR / "training_data" / "processed"MANIFEST   = UPSTREAM_DIR / "configs" / f"data_{SURFACE}.json"EXP_DIR    = UPSTREAM_DIR / "examples" / "acdc" / "4DMM" / SURFACE# --- Geometry (must match the validated baseline notebook) ------------------------RV_LABEL             = 1      # get_P.py convention: 1=RV, 2=LVM, 3=LVMARCHING_CUBES_LEVEL = 0.5MIN_COMPONENT_VOXELS = 32SMOOTHING_ITERATIONS = 1# --- SDF sampling: values copied from upstream process_data.py:process_frame -------SDF_SAMPLE_COUNT     = 250000SURFACE_SAMPLE_COUNT = 10000000NORMAL_SAMPLE_COUNT  = 5        # upstream uses 5, not the library default 11ED_INDEX = 0   # production treats ED as SUPPLIED (FourDReconstructionJobRequest.               # ed_frame_index). The baseline notebook showed argmax-ED sits inside the               # measurement error bar on patient005, so it is not derived here either.print("config ok")

In [ ]:
if not UPSTREAM_DIR.exists():    subprocess.run(["git", "clone", "--depth", "1", UPSTREAM_URL, str(UPSTREAM_DIR)], check=True)print("upstream at", UPSTREAM_DIR)# NOTE: the repo ships "4DM Data Access Agreement.pdf". Read it before using the bundled# ACDC-derived assets or redistributing anything trained on them.for p in (str(UPSTREAM_DIR), str(DEPS_DIR)):    if p not in sys.path:        sys.path.insert(0, p)import numpy as np, torch, trimesh, nibabel as nibfrom skimage import measurefrom scipy import ndimagefrom mesh_to_sdf.mesh_to_sdf import sample_sdf_near_surface, transformationfrom get_P import get_T                      # production canonical pose, not a copyDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")print("torch", torch.__version__, "| device:", DEVICE)

## 4. Build the RV training corpusTarget format, from upstream `process_data.py`:- one `.npz` per frame, keys `TRAIN_KEYS = {pos, neg, t, P, Pi, offset, scale}`- `pos` = `[xyz, sdf]` where `sdf > 0`; `neg` where `sdf < 0` (strict — zeros dropped)- `t = phase_index / max(frame_count - 1, 1)`, so phase 0 (ED) gives `t = 0`- manifest `{"train": {"<dataset>": {"<seq>": {"instance_list": [...]}}}, "test": {}}`Two RV-specific deviations, both flagged:1. Upstream reads `P/Pi/offset/scale` from a per-patient transform `.txt` produced by its own   `preprocess_mask.py`. Here they come from production's `get_T` instead, so the training   canonical frame is identical to the one inference uses. **`P <-> T` and `Pi <-> Ti` is an   inferred correspondence, not verified** — `Pi` is reconstructed the way the handler builds   `Ti` (`fourdreconstruction_handler.py:437-439`). Confirm against `reconstruct_4dmm.py`   before trusting reconstructed meshes in world space.2. Meshes come from the validated baseline pipeline rather than upstream's OBJs. Sampling   signs correctly only on watertight, consistently-wound input — 30/30 watertight with zero   non-manifold edges was measured on patient005, which is the precondition here, re-asserted   per frame rather than assumed.

In [ ]:
import tempfile, shutildef load_label_volume(nifti_path):    img = nib.load(str(nifti_path))    data = np.asarray(img.dataobj)    if data.ndim != 4:        raise ValueError(f"expected 4D (X,Y,Z,T), got {data.shape}")    return data, img.affinedef keep_main_component(binary, min_voxels=MIN_COMPONENT_VOXELS):    lab, n = ndimage.label(binary)    if n == 0:        return binary    sizes = ndimage.sum(binary, lab, range(1, n + 1))    keep = np.arange(1, n + 1)[sizes >= min_voxels]    if keep.size == 0:        keep = np.array([int(np.argmax(sizes)) + 1])    return np.isin(lab, keep)def frame_to_mesh(vol_frame, affine):    # Same pipeline as the validated baseline notebook (Task 4.3).    binary = keep_main_component(vol_frame == RV_LABEL)    binary = ndimage.binary_closing(binary, iterations=1)    if binary.sum() < MIN_COMPONENT_VOXELS:        return None    padded = np.pad(binary.astype(np.float32), 1, mode="constant")    verts, faces, _, _ = measure.marching_cubes(padded, level=MARCHING_CUBES_LEVEL)    verts -= 1.0    world = (np.c_[verts, np.ones(len(verts))] @ affine.T)[:, :3]   # NIfTI world coords    mesh = trimesh.Trimesh(vertices=world, faces=faces, process=False)    if SMOOTHING_ITERATIONS > 0:        trimesh.smoothing.filter_taubin(mesh, iterations=SMOOTHING_ITERATIONS)    return meshdef canonical_from_nifti_frame(vol_frame, affine):    # get_T needs an on-disk <root>/<patient>/<file> layout; reuse the handler's workaround    # (fourdreconstruction_handler.py:394-406) rather than reimplementing it.    with tempfile.TemporaryDirectory() as tmp:        pdir = os.path.join(tmp, "patient001")        os.makedirs(pdir, exist_ok=True)        fn = "frame.nii.gz"        nib.save(nib.Nifti1Image(vol_frame.astype(np.int16), affine), os.path.join(pdir, fn))        T, offset, scale = get_T(tmp, "acdc", "patient001", fn)    Ti = np.identity(4)    Ti[0:3, 0:3] = np.linalg.inv(T[0:3, 0:3])    Ti[0:3, 3] = -np.dot(np.linalg.inv(T[0:3, 0:3]), T[0:3, 3])    return T, Ti, offset, scaledef to_canonical(mesh, T, offset, scale):    v = (transformation(T, mesh.vertices.transpose()) + offset) * scale    return trimesh.Trimesh(vertices=v, faces=mesh.faces, process=False)print("helpers defined")

In [ ]:
def build_sequence(nifti_path, seq_name, ed_index=ED_INDEX):    PROC_ROOT.mkdir(parents=True, exist_ok=True)    vol4d, affine = load_label_volume(nifti_path)    n_src = vol4d.shape[3]    # Rotate so ED is phase 0 => t == 0 marks ED, which is what Deformation keys off.    order = [(ed_index + i) % n_src for i in range(n_src)]    # Canonical frame is computed ONCE, from ED, and reused for every phase. Recomputing per    # frame would let the canonical pose absorb the deformation the motion model must learn.    # NOTE (4.1b, still open): get_T derives pose from LVM(2) and needs LV labels present in    # the volume -- it is not RV-only. A mask with RV but no LVM will raise here. The    # geometric baseline never exercised this path; this is the first code that does.    T, Ti, offset, scale = canonical_from_nifti_frame(vol4d[..., ed_index], affine)    print(f"{seq_name}: {n_src} frames, ED={ed_index}, scale={float(np.ravel(scale)[0]):.6f}")    written = []    for phase, src_i in enumerate(order):        mesh = frame_to_mesh(vol4d[..., src_i], affine)        if mesh is None:            print(f"  [skip] phase {phase} (src {src_i}): empty RV"); continue        if not mesh.is_watertight:            print(f"  [WARN] phase {phase} (src {src_i}): NOT watertight -> skipped"); continue        cmesh = to_canonical(mesh, T, offset, scale)        np.random.seed((hash((seq_name, phase)) & 0x7FFFFFFF) % (2**32 - 1))        pts, sdf = sample_sdf_near_surface(            cmesh,            number_of_points=SDF_SAMPLE_COUNT,            sampling_type="sphere",            surface_point_method="sample",            sign_method="normal",            scan_count=0,            scan_resolution=0,            sample_point_count=SURFACE_SAMPLE_COUNT,            normal_sample_count=NORMAL_SAMPLE_COUNT,            min_size=0,            return_gradients=False,            test_sampling=False,        )        pos = np.column_stack((pts[sdf > 0], sdf[sdf > 0])).astype(np.float32)        neg = np.column_stack((pts[sdf < 0], sdf[sdf < 0])).astype(np.float32)        if len(pos) == 0 or len(neg) == 0:            print(f"  [WARN] phase {phase}: degenerate signs "                  f"(pos={len(pos)}, neg={len(neg)}) -> skipped"); continue        t_val = np.float32(phase / max(len(order) - 1, 1))        out = PROC_ROOT / f"{seq_name}_{phase:02d}-{SURFACE}.npz"        np.savez_compressed(out, pos=pos, neg=neg, t=t_val, P=T, Pi=Ti,                            offset=np.asarray(offset).reshape(-1),                            scale=np.asarray(scale).reshape(-1))        written.append(out)    print(f"  wrote {len(written)}/{n_src} frames")    return writtensequences = {}for p in MASK_PATHS:    name = Path(p).name.split(".")[0]    sequences[name] = build_sequence(p, seq_name=name)

In [ ]:
# Manifest: paths relative to the upstream repo root, matching resolve_manifest_paths().manifest = {"train": {"training": {}}, "test": {}}for name, files in sequences.items():    manifest["train"]["training"][name] = {        "instance_list": [f.resolve().relative_to(UPSTREAM_DIR.resolve()).as_posix()                          for f in files]    }MANIFEST.parent.mkdir(parents=True, exist_ok=True)MANIFEST.write_text(json.dumps(manifest, indent=2) + "\n")n_seq = len(manifest["train"]["training"])n_frames = sum(len(s["instance_list"]) for s in manifest["train"]["training"].values())print(f"{MANIFEST}\n  {n_seq} sequence(s), {n_frames} frames")if n_seq < 5:    print("[!] Far below any published cardiac DeepSDF corpus (SDF4CHD: 67 real scans, "          "augmented to 420). Treat results as a plumbing test, not a model. Report "          "per-patient variance, not the mean -- high variance is the small-N failure "          "signature (vanilla DeepSDF RV Dice 91.4+/-17.0 vs regularized 94.1+/-3.2).")

## 5. RV specsCopied from upstream `examples/acdc/4DMM/endo/specs.json`, changing nothing but the datasource. The network shape is held identical to endo/epi on purpose: the plan's assumption isthat only the checkpoint needs to differ for RV, and changing architecture and data at oncewould make a failure uninterpretable.Note the real hyperparameters — `NumEpochs: 50`, `BatchSize: 16`, `SamplesPerScene: 8000`,four separate LR schedules (decoder x2, `c_s`, `c_m`), `CodeBound: 1.0` (applied as`Embedding(max_norm=...)`), pointwise 5e-4 and pointpair 1e-4 with opposite epoch annealing.`LipschitzLoss(k=0.5)` is hardcoded in `train_4dmm.py`, not exposed in specs.

In [ ]:
RV_SPECS = json.loads((UPSTREAM_DIR / "examples/acdc/4DMM/endo/specs.json").read_text())RV_SPECS["Description"] = [f"RV cavity 4DMM ({SURFACE}) - VisHeart Task 4.7"]RV_SPECS["DataSource"] = str(MANIFEST.relative_to(UPSTREAM_DIR).as_posix())EXP_DIR.mkdir(parents=True, exist_ok=True)(EXP_DIR / "specs.json").write_text(json.dumps(RV_SPECS, indent=2) + "\n")print("wrote", EXP_DIR / "specs.json")print({k: RV_SPECS[k] for k in       ("NetworkArch", "NumEpochs", "BatchSize", "SamplesPerScene",        "ClampingDistance", "CodeBound", "PointwiseLossWeight", "PointpairLossWeight")})

## 6. TrainUpstream `train_4dmm.run_training(surface=...)`. Smoke first (`smoke_test=True` drops`SamplesPerScene` to 256), and only then the real run.Registering `rv` as a surface may need an entry in `configs/training.json` and`project_config.py`, which currently enumerate `endo` and `epi`. If `run_training` rejectsthe surface name, that is the reason — add `rv` there rather than renaming the RV data to`endo`, which would silently overwrite the shipped ACDC checkpoint.

In [ ]:
# from train_4dmm import run_training## run_training(surface=SURFACE, device=DEVICE, smoke_test=True)     # <- do this first# run_training(surface=SURFACE, device=DEVICE)                      # full runprint("Training call is commented out on purpose -- see Section 8.")

## 7. Production compatibility assertion

In [ ]:
def verify_production_loadable(ckpt_path):    # Reproduce the handler's load path exactly (fourdreconstruction_handler.py:113-129),    # including its OWN specs, to prove the upstream-trained checkpoint drops into    # production without touching either source tree.    prod_specs = {        "NetworkArch": "decoder",        "NetworkSpecs": {            "motionmodel_kargs": {"dim": 4, "in_features": 256,                                  "out_features": 3, "num_filters": 32},            "shapemodel_kargs": {                "latent_size": 256, "dims": [512] * 8, "dropout": list(range(8)),                "dropout_prob": 0.2, "norm_layers": list(range(8)), "latent_in": [4],                "xyz_in_all": False, "use_tanh": False, "latent_dropout": False,                "weight_norm": True,            },        },    }    arch = __import__("networks." + prod_specs["NetworkArch"], fromlist=["Decoder"])    probe = arch.Decoder(**prod_specs["NetworkSpecs"])    saved = torch.load(ckpt_path, map_location="cpu")    assert set(saved.keys()) >= {"epoch", "model_state_dict"}, saved.keys()    probe.load_state_dict(saved["model_state_dict"])   # strict=True on purpose    probe.eval()    size_mb = os.path.getsize(ckpt_path) / 1024**2    print(f"OK: {Path(ckpt_path).name} loads into production Decoder "          f"(epoch {saved['epoch']}, {size_mb:.2f} MB)")    return probe# Already executed outside this notebook against upstream's shipped endo checkpoint: PASSED# (strict=True, epoch 44). Kept runnable to re-check after RV training.verify_production_loadable(UPSTREAM_DIR / "examples/acdc/4DMM/endo/ModelParameters/latest.pth")

## 8. What has not been doneNothing above has been executed. In order:1. ~~Run the Section 7 assertion first.~~ **Done** — executed outside the notebook and it   passes (upstream `endo` checkpoint loads into production's `Decoder` under `strict=True`,   epoch 44). The premise of this notebook holds. Re-run it against the RV checkpoint once   training produces one.2. **`get_T` on RV data is unproven.** It derives canonical pose from LVM(2), so it requires   LV labels in the same volume and will raise on an RV-only mask. This is task 4.1b, still   open, and Section 4 is the first code path that exercises it.3. **`P`/`Pi` correspondence is inferred, not verified.** Cross-check against   `reconstruct_4dmm.py` before believing any world-space mesh that comes back out.4. **Sign correctness of the sampled SDF is unverified.** Watertight is asserted, winding is   not. Cheapest check: take points known to be inside the cavity, confirm `sdf < 0`.5. **Surface registration.** `rv` may need adding to `configs/training.json` and   `project_config.py`. Do not reuse the `endo` slot.6. **Corpus size.** One patient proves the loop runs and nothing more. The 2-3 masks in   progress bring it to 3-4. Frame the Week-2 go/no-go as *"is training stable at this N"*,   with per-patient variance reported — not *"did it converge"*.7. **No accuracy metric is wired.** `deep_sdf/metrics/chamfer.py` exists; the plan's bar is   Chamfer < ~4 mm, Hausdorff < ~10 mm. Chamfer must be computed in **millimetres** — undo   `scale`/`offset` first — or the number cannot be compared to that bar at all.8. **Motion model unreachable from production inference** (Section 2a). Team decision needed:   does Week 2 also deliver the inference-side change, or does the checkpoint ship shape-only?